# Steam Game Stats & Reddit Hype Correlation
**Social Computing - 2026**

### Group Members:
* **Data Collection:** Antonio & Jacob
* **Data Analysis:** Elias & Alina

**Reference:** Presentation: "Steam Game Stats & Reddit Hype" 


```⮕  The markdown cells were originally generated with Gemini based on our Research Proposal Presentation Slides: "Steam Game Stats & Reddit Hype"```


## 1. Motivation
Understanding social media dynamics is valuable for game studios, publishers, and developers to:
* Determine if a game's success is due to **long-term retention** or **short-term hype**.
* Evaluate if "hype-focused" launch strategies or general marketing are worth the investment compared to traditional development.
* Link public social signals to behavioral outcomes (e.g., playtime and sales).
## 2. Research Framework
### Research Questions
* **RQ1:** Are the number of user recommendations higher when engagement on Reddit is high?
* **RQ2:** Does Reddit engagement correlate with actual time spent on the game?

In [16]:
import pandas as pd
import os
from better_profanity import profanity

repo_root = os.getcwd()
file_path = os.path.join(repo_root, "Steam Dataset", "games.json")

df = pd.read_json(file_path, orient='index')

print(f"Full Dataset: Total games: {len(df)}")
df.head()


#example subset of dataset:
df_2021 = df[df['release_date'].str.contains('2021', na=False)].copy()
print(f"2021 Dataset: Total games: {len(df_2021)}")

EXCLUDED_LABELS = {"Sexual Content", "Nudity"}

def is_excluded(game):
    langs = game.get('supported_languages') or []
    if 'English' not in langs:
        return True
    name = (game.get('name') or '').lower()
    description = (game.get('short_description') or '').lower()
    if profanity.contains_profanity(name) or profanity.contains_profanity(description):
        return True
    tags = set(game.get('tags') or {})
    genres = set(game.get('genres') or [])
    return bool((tags | genres) & EXCLUDED_LABELS)


df_2021 = df[df['release_date'].str.contains('2021', na=False)].copy()
mask = df_2021.apply(is_excluded, axis=1)


df_2021_clean = df_2021[~mask]
df_2021_removed = df_2021[mask]

df_2021_clean.to_csv('games_2021_filtered.csv')
df_2021_removed.to_csv('games_2021_removed.csv')

print(f"Clean games: {len(df_2021_clean)}")
print(f"Removed games: {len(df_2021_removed)}")

Full Dataset: Total games: 122611
2021 Dataset: Total games: 11067
Clean games: 8704
Removed games: 2363


In [18]:
df_2021_clean['genres'].explode().value_counts()

genres
Indie                    6242
Casual                   4017
Action                   3634
Adventure                3552
Simulation               1710
Strategy                 1688
RPG                      1363
Early Access              856
Free To Play              828
Sports                    386
Racing                    345
Massively Multiplayer     174
Utilities                  89
Design & Illustration      45
Education                  39
Game Development           36
Animation & Modeling       35
Video Production           31
Audio Production           18
Photo Editing              17
Software Training          15
Web Publishing             10
Accounting                  1
Name: count, dtype: int64

Clean Action Games: 0


### Hypotheses
* **H1:** There are more user recommendations when engagement on Reddit is high.
* **H2:** For Buy-to-Play (B2P) games, median playtime is lower when Reddit engagement is high.
* **H3:** Among high-engagement games, Free-to-Play (F2P) titles have lower median playtime than B2P titles.

## 3. Data Collection Methodology
**Assigned to: Antonio & Jacob**


### Steam Data (Kaggle Dataset)
We are extracting the following features:
* `name`: Game title
* `recommendations`: User recommendation count
* `average_playtime_forever` & `median_playtime_forever`
* `price`: To distinguish between F2P (0.0) and B2P games
* `peak_ccu`: Peak concurrent users

### Reddit Engagement Data (PRAW)
Using the **Python Reddit API Wrapper (PRAW)** to scrape `r/gaming`:
* **Target:** "Top" posts where the game name is featured in the title.
* **Engagement Metric:** Sum of upvotes.
* **Valence Metric:** Upvote/downvote ratio of the top 50 posts.


## 4. Methods of Analysis
**Assigned to: Elias & Alina**

We will employ three primary statistical methods:

1. **Spearman Rank Correlation:** Chosen to handle outliers and right-skewed gaming data. We will test the correlation between upvotes and recommendations (H1) and upvotes and playtime (H2).
2. **OLS Regression:** To measure the independent influence of variables (upvotes, valence, price) on playtime and player count.
3. **Mann-Whitney U-test:** To compare distributions between groups (F2P vs. B2P) without assuming normal distribution (H3).